In [32]:
import pymongo
from pymongo import MongoClient
from pymongo.errors import ServerSelectionTimeoutError, PyMongoError
from google.colab import userdata

# Get MongoDB link from Colab secret
mongo_link = userdata.get("MONGO_CONNECTION_STRING")

try:
    client = MongoClient(mongo_link)

    # The ping command is cheap and does not require auth.
    client.admin.command('ping')
    print("MongoDB connection successful!")

except ServerSelectionTimeoutError as err:
    print(f"MongoDB connection failed: Server Selection Timeout. This often indicates network issues or incorrect IP whitelisting in MongoDB Atlas. Error: {err}")
except PyMongoError as err:
    print(f"An unexpected PyMongo error occurred during connection: {err}")
except Exception as err:
    print(f"An unexpected error occurred: {err}")


MongoDB connection successful!


In [33]:
# Unzip the dataset first
!unzip -o northstar_dataset.zip

print("Dataset unzipped")


Archive:  northstar_dataset.zip
  inflating: __MACOSX/._northstar_dataset  
  inflating: northstar_dataset/hubs.csv  
  inflating: __MACOSX/northstar_dataset/._hubs.csv  
  inflating: northstar_dataset/customers.csv  
  inflating: __MACOSX/northstar_dataset/._customers.csv  
  inflating: northstar_dataset/orders.csv  
  inflating: __MACOSX/northstar_dataset/._orders.csv  
  inflating: northstar_dataset/complaints.csv  
  inflating: __MACOSX/northstar_dataset/._complaints.csv  
  inflating: northstar_dataset/drivers.csv  
  inflating: __MACOSX/northstar_dataset/._drivers.csv  
  inflating: northstar_dataset/deliveries.csv  
  inflating: __MACOSX/northstar_dataset/._deliveries.csv  
  inflating: northstar_dataset/README.txt  
  inflating: __MACOSX/northstar_dataset/._README.txt  
  inflating: northstar_dataset/app_events.csv  
  inflating: __MACOSX/northstar_dataset/._app_events.csv  
  inflating: northstar_dataset/data_dictionary.csv  
  inflating: __MACOSX/northstar_dataset/._data_dict

In [34]:
# Load data for MongoDB section

complaints = pd.read_csv("northstar_dataset/complaints.csv")
incidents = pd.read_csv("northstar_dataset/incidents.csv")
deliveries = pd.read_csv("northstar_dataset/deliveries.csv")
vehicles = pd.read_csv("northstar_dataset/vehicles.csv")
app_events = pd.read_csv("northstar_dataset/app_events.csv")
orders = pd.read_csv("northstar_dataset/orders.csv")

print("Data loaded")

print("Complaints:", len(complaints))
print("Incidents:", len(incidents))
print("Deliveries:", len(deliveries))
print("Vehicles:", len(vehicles))
print("App events:", len(app_events))
print("Orders:", len(orders))


Data loaded
Complaints: 320
Incidents: 280
Deliveries: 950
Vehicles: 120
App events: 640
Orders: 1250


In [35]:
# Insert data into customer_cases collection

# Define the database and collection
db = client.northstar_data
customer_cases = db.customer_cases

# Clear old data first
customer_cases.drop()

# Add one sample complaint document
sample_complaint = {
    "complaint_id": "CP_SAMPLE",
    "customer_id": "C0001",
    "order_id": "O00001",
    "complaint_type": "Delay",
    "severity": "High",
    "channel": "App",
    "status": "Open",
    "created_at": "2025-11-14 09:22:00",
    "resolution_days": None,
    "compensation_amount": 37.50,
    "complaint_events": [
        {
            "event_type": "complaint_created",
            "timestamp": "2025-11-14 09:22:00",
            "notes": "Customer complained about late delivery"
        },
        {
            "event_type": "complaint_checked",
            "timestamp": "2025-11-14 10:05:00",
            "notes": "Complaint sent to support team"
        }
    ]
}

customer_cases.insert_one(sample_complaint)

# Add some complaints from the CSV file
complaint_list = []

for index, row in complaints.head(10).iterrows():
    complaint = {
        "complaint_id": row["complaint_id"],
        "customer_id": row["customer_id"],
        "order_id": row["order_id"],
        "complaint_type": row["complaint_type"],
        "severity": row["severity"],
        "channel": row["channel"],
        "status": row["status"],
        "created_at": row["created_at"],
        "resolution_days": row["resolution_days"],
        "compensation_amount": row["compensation_amount"],
        "complaint_events": [
            {
                "event_type": "complaint_created",
                "timestamp": row["created_at"],
                "notes": "Complaint came from " + str(row["channel"])
            }
        ]
    }

    complaint_list.append(complaint)

customer_cases.insert_many(complaint_list)

print("Customer complaint documents inserted")
print("Total documents:", customer_cases.count_documents({}))

Customer complaint documents inserted
Total documents: 11


In [36]:
# Insert data into the other collections

# Define the database and collections
db = client.northstar_data
driver_events = db.driver_events
route_exceptions = db.route_exceptions
vehicle_health = db.vehicle_health
app_interactions = db.app_interactions

# Clear old data first
driver_events.drop()
route_exceptions.drop()
vehicle_health.drop()
app_interactions.drop()

# 1. Driver events from incidents table
driver_list = []

for index, row in incidents.head(15).iterrows():
    driver_doc = {
        "incident_id": row["incident_id"],
        "delivery_id": row["delivery_id"],
        "incident_type": row["incident_type"],
        "reported_at": row["reported_at"],
        "severity": row["severity"],
        "resolution_status": row["resolution_status"],
        "resolved_hours": row["resolved_hours"],
        "incident_info": {
            "needs_checking": row["severity"] == "High",
            "notes": "Incident reported for delivery"
        }
    }

    driver_list.append(driver_doc)

driver_events.insert_many(driver_list)
print("Driver event documents inserted:", driver_events.count_documents({}))


# 2. Route exceptions from failed and delayed deliveries
problem_deliveries = deliveries[
    deliveries["delivery_status"].isin(["Failed", "Delayed"])
].head(15)

route_list = []

for index, row in problem_deliveries.iterrows():
    route_doc = {
        "delivery_id": row["delivery_id"],
        "order_id": row["order_id"],
        "driver_id": row["driver_id"],
        "delivery_status": row["delivery_status"],
        "route_distance_km": row["route_distance_km"],
        "fuel_or_charge_cost": row["fuel_or_charge_cost"],
        "manual_route_changes": row["manual_route_override_count"],
        "extra_details": {
            "proof_missing": bool(row["proof_of_completion_missing"]),
            "customer_rating": row["customer_rating_post_delivery"]
        }
    }

    route_list.append(route_doc)

route_exceptions.insert_many(route_list)
print("Route exception documents inserted:", route_exceptions.count_documents({}))


# 3. Vehicle health documents
vehicle_list = []

for index, row in vehicles.iterrows():
    vehicle_doc = {
        "vehicle_id": row["vehicle_id"],
        "vehicle_type": row["vehicle_type"],
        "assigned_zone": row["assigned_zone"],
        "commission_date": row["commission_date"],
        "health": {
            "battery_health_pct": row["battery_health_pct"],
            "odometer_km": row["odometer_km"],
            "maintenance_status": row["maintenance_status"]
        }
    }

    vehicle_list.append(vehicle_doc)

vehicle_health.insert_many(vehicle_list)
print("Vehicle health documents inserted:", vehicle_health.count_documents({}))


# 4. App interaction documents
app_list = []

for index, row in app_events.head(20).iterrows():
    app_doc = {
        "event_id": row["event_id"],
        "customer_id": row["customer_id"],
        "order_id": row["order_id"],
        "event_details": {
            "event_type": row["event_type"],
            "event_timestamp": row["event_timestamp"],
            "session_id": row["session_id"],
            "device_type": row["device_type"],
            "zone_context": row["zone_context"],
            "api_latency_ms": row["api_latency_ms"],
            "success_flag": bool(row["success_flag"])
        }
    }

    app_list.append(app_doc)

app_interactions.insert_many(app_list)
print("App interaction documents inserted:", app_interactions.count_documents({}))

print("All collections have data now")

Driver event documents inserted: 15
Route exception documents inserted: 15
Vehicle health documents inserted: 120
App interaction documents inserted: 20
All collections have data now


In [37]:
# Read data from MongoDB collections

print("Read examples")

# 1. Find one high severity complaint
print("\nOne high severity complaint")
one_complaint = customer_cases.find_one({"severity": "High"})
pprint(one_complaint)

# 2. Find complaints that took 10 or more days
print("\nComplaints with 10 or more resolution days")
long_complaints = customer_cases.find(
    {"resolution_days": {"$gte": 10}},
    {
        "complaint_id": 1,
        "complaint_type": 1,
        "severity": 1,
        "resolution_days": 1,
        "_id": 0
    }
)

for item in long_complaints:
    print(item)

# 3. Find open complaints
print("\nOpen complaints")
open_complaints = customer_cases.find(
    {"status": "Open"},
    {
        "complaint_id": 1,
        "severity": 1,
        "status": 1,
        "_id": 0
    }
)

for item in open_complaints:
    print(item)

# 4. Count high severity complaints
high_count = customer_cases.count_documents({"severity": "High"})
print("\nHigh severity complaint count:", high_count)

# 5. Find failed route exceptions
print("\nFailed route exceptions")
failed_routes = route_exceptions.find(
    {"delivery_status": "Failed"},
    {
        "delivery_id": 1,
        "order_id": 1,
        "driver_id": 1,
        "delivery_status": 1,
        "fuel_or_charge_cost": 1,
        "_id": 0
    }
).limit(5)

for item in failed_routes:
    print(item)


Read examples

One high severity complaint
{'_id': ObjectId('6a061cf623541b64bb5bc24d'),
 'channel': 'App',
 'compensation_amount': 37.5,
 'complaint_events': [{'event_type': 'complaint_created',
                       'notes': 'Customer complained about late delivery',
                       'timestamp': '2025-11-14 09:22:00'},
                      {'event_type': 'complaint_checked',
                       'notes': 'Complaint sent to support team',
                       'timestamp': '2025-11-14 10:05:00'}],
 'complaint_id': 'CP_SAMPLE',
 'complaint_type': 'Delay',
 'created_at': '2025-11-14 09:22:00',
 'customer_id': 'C0001',
 'order_id': 'O00001',
 'resolution_days': None,
 'severity': 'High',
 'status': 'Open'}

Complaints with 10 or more resolution days
{'complaint_id': 'CP0001', 'complaint_type': 'AppIssue', 'severity': 'High', 'resolution_days': 11}
{'complaint_id': 'CP0003', 'complaint_type': 'Delay', 'severity': 'High', 'resolution_days': 16}
{'complaint_id': 'CP0008', 'compl

In [38]:
# Update data in MongoDB

print("Update examples")

# Check total complaints
print("Total complaints:", customer_cases.count_documents({}))

# Update one complaint
update1 = customer_cases.update_one(
    {"complaint_id": "CP_SAMPLE"},
    {
        "$set": {
            "status": "Resolved",
            "resolution_days": 3,
            "resolved_at": "2025-11-17 10:00:00"
        }
    }
)

print("\nOne complaint updated")
print("Matched:", update1.matched_count)
print("Changed:", update1.modified_count)

# Check the updated complaint
updated_complaint = customer_cases.find_one(
    {"complaint_id": "CP_SAMPLE"},
    {
        "complaint_id": 1,
        "status": 1,
        "resolution_days": 1,
        "_id": 0
    }
)

print(updated_complaint)

# Update many low severity open complaints
update2 = customer_cases.update_many(
    {"severity": "Low", "status": "Open"},
    {
        "$set": {
            "status": "Under Review"
        }
    }
)

print("\nLow severity open complaints updated")
print("Matched:", update2.matched_count)
print("Changed:", update2.modified_count)

# Update vehicles with low battery
update3 = vehicle_health.update_many(
    {"health.battery_health_pct": {"$lt": 40}},
    {
        "$set": {
            "health.maintenance_status": "Urgent"
        }
    }
)

print("\nLow battery vehicles updated")
print("Matched:", update3.matched_count)
print("Changed:", update3.modified_count)


Update examples
Total complaints: 11

One complaint updated
Matched: 1
Changed: 1
{'complaint_id': 'CP_SAMPLE', 'status': 'Resolved', 'resolution_days': 3}

Low severity open complaints updated
Matched: 0
Changed: 0

Low battery vehicles updated
Matched: 0
Changed: 0


In [39]:
# Delete example in MongoDB

print("Delete example")

# Count before adding test data
before_count = customer_cases.count_documents({})
print("Documents before:", before_count)

# Add a test document
test_doc = {
    "complaint_id": "TEST_DELETE",
    "customer_id": "C9999",
    "complaint_type": "Test",
    "severity": "Low",
    "status": "Resolved",
    "resolution_days": 1
}

customer_cases.insert_one(test_doc)

print("Documents after adding test document:", customer_cases.count_documents({}))

# Delete the test document
delete_result = customer_cases.delete_one({"complaint_id": "TEST_DELETE"})

print("Deleted documents:", delete_result.deleted_count)

# Count after deleting
after_count = customer_cases.count_documents({})
print("Documents after delete:", after_count)

# Check if it is deleted
check_delete = customer_cases.find_one({"complaint_id": "TEST_DELETE"})
print("Check deleted document:", check_delete)


Delete example
Documents before: 11
Documents after adding test document: 12
Deleted documents: 1
Documents after delete: 11
Check deleted document: None


In [40]:
# Aggregation 1: Complaints by severity

pipeline1 = [
    {
        "$group": {
            "_id": "$severity",
            "total_complaints": {"$sum": 1},
            "average_resolution_days": {"$avg": "$resolution_days"},
            "average_compensation": {"$avg": "$compensation_amount"}
        }
    },
    {
        "$sort": {
            "total_complaints": -1
        }
    }
]

results1 = list(customer_cases.aggregate(pipeline1))

print("Complaints by severity")

for item in results1:
    print(item)


Complaints by severity
{'_id': 'Medium', 'total_complaints': 6, 'average_resolution_days': 4.333333333333333, 'average_compensation': 16.705000000000002}
{'_id': 'High', 'total_complaints': 4, 'average_resolution_days': 12.0, 'average_compensation': nan}
{'_id': 'Low', 'total_complaints': 1, 'average_resolution_days': 4.0, 'average_compensation': 26.35}


In [41]:
# Aggregation 2: Route exceptions by delivery status

pipeline2 = [
    {
        "$group": {
            "_id": "$delivery_status",
            "total_exceptions": {"$sum": 1},
            "average_cost": {"$avg": "$fuel_or_charge_cost"},
            "average_distance": {"$avg": "$route_distance_km"},
            "average_route_changes": {"$avg": "$manual_route_changes"}
        }
    },
    {
        "$sort": {
            "total_exceptions": -1
        }
    }
]

results2 = list(route_exceptions.aggregate(pipeline2))

print("Route exceptions summary")

for item in results2:
    print(item)


Route exceptions summary
{'_id': 'Failed', 'total_exceptions': 9, 'average_cost': 13.637777777777778, 'average_distance': 13.483333333333333, 'average_route_changes': 1.2222222222222223}
{'_id': 'Delayed', 'total_exceptions': 6, 'average_cost': 13.168333333333331, 'average_distance': 18.996666666666666, 'average_route_changes': 0.3333333333333333}


In [42]:
# Aggregation 3: App event success

pipeline3 = [
    {
        "$group": {
            "_id": "$event_details.event_type",
            "total_events": {"$sum": 1},
            "successful_events": {
                "$sum": {
                    "$cond": [
                        {"$eq": ["$event_details.success_flag", True]},
                        1,
                        0
                    ]
                }
            },
            "average_latency": {"$avg": "$event_details.api_latency_ms"}
        }
    },
    {
        "$addFields": {
            "success_percentage": {
                "$multiply": [
                    {"$divide": ["$successful_events", "$total_events"]},
                    100
                ]
            }
        }
    },
    {
        "$sort": {
            "success_percentage": 1
        }
    }
]

results3 = list(app_interactions.aggregate(pipeline3))

print("App event success summary")

for item in results3:
    print(item)


App event success summary
{'_id': 'chat_escalated', 'total_events': 3, 'successful_events': 1, 'average_latency': 361.0, 'success_percentage': 33.33333333333333}
{'_id': 'track_order', 'total_events': 3, 'successful_events': 3, 'average_latency': 734.6666666666666, 'success_percentage': 100.0}
{'_id': 'delivery_instruction_update', 'total_events': 2, 'successful_events': 2, 'average_latency': 404.0, 'success_percentage': 100.0}
{'_id': 'search_route', 'total_events': 4, 'successful_events': 4, 'average_latency': 271.75, 'success_percentage': 100.0}
{'_id': 'chat_opened', 'total_events': 5, 'successful_events': 5, 'average_latency': 377.6, 'success_percentage': 100.0}
{'_id': 'eta_refresh', 'total_events': 3, 'successful_events': 3, 'average_latency': 450.6666666666667, 'success_percentage': 100.0}


In [43]:
# Final collection count

collections = {
    "customer_cases": customer_cases,
    "driver_events": driver_events,
    "route_exceptions": route_exceptions,
    "vehicle_health": vehicle_health,
    "app_interactions": app_interactions
}

total_documents = 0

print("Final MongoDB collection summary")

for name, collection in collections.items():
    count = collection.count_documents({})
    total_documents = total_documents + count
    print(name, ":", count)

print("Total documents:", total_documents)
print("Section 4 finished")


Final MongoDB collection summary
customer_cases : 11
driver_events : 15
route_exceptions : 15
vehicle_health : 120
app_interactions : 20
Total documents: 181
Section 4 finished
